# Method B - finish the two remaining 1.5B main runs

Open on **Sol in a fresh one-A100 GPU Jupyter session**, using Python 3.
Use the same A100 80 GB type and your existing project environment. A fresh
**3-hour allocation** gives a margin; measured expected runtime is **1-1.5 hours**.
Cache regeneration, retries, or slow storage can take longer.

This notebook schedules **only**:

| Model | Original parent seed | B training seed | Setting / budget |
|---|---|---|---|
| Qwen 1.5B | 17 | 2026 | Mixed fresh-random B, 1x |
| Qwen 1.5B | 123 | 2026 | Mixed fresh-random B, 1x |

Every run starts independently from its matching original R1000 parent. Previously
completed work is reused only when its saved completion evidence passes the existing
checks. The selected dataset, model revisions, training recipe, and evaluator remain
the existing version. No transfer or 0.5x/2x/4x cases are scheduled.

**Run the code cells in order** with Shift+Enter, or use Run All Cells. Cell 4 monitors
the background worker until it exits. You can interrupt that monitoring cell and
rerun it later; this does not stop the detached worker. Keep the Sol allocation active.
An allocation ending stops its processes even if the browser tab remains open.

The worker uses the existing `Method_B_Overnight.py` as a checked backend; it never
starts that file's full queue. Duplicate project workers are blocked by the same
worker/evaluation locks. Failed stages retry once when time allows; independent
cases can continue. Progress, exceptions and results are saved outside the notebook.

This finishes the two remaining **main development** cases, not transfer or protected
final evaluation. Do not run the old overnight notebook or runner at the same time.


## Cell 1 - project settings

The notebook calls your existing training Python even if the kernel uses a different Python 3 environment.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import fcntl
import hashlib
import json
import os
import subprocess
import time
import uuid

ROOT = Path('/scratch/sshind38/capstone_approach_b')
TRAINING_PYTHON = ROOT / 'venv_b_training_py312/bin/python'
WORKER_HOURS = 2.75
AREA = ROOT / 'outputs/B_method_B_document_v3/finish_1p5B_seed2026'
if not ROOT.is_dir() or not TRAINING_PYTHON.is_file():
    raise FileNotFoundError('Open this notebook on Sol, where the existing project and training environment are available.')
backend = ROOT / 'Method_B_Overnight.py'
if hashlib.sha256(backend.read_bytes()).hexdigest() != 'd8101467d2037095bffdfb851959c2a2e2405daeb2de377b703ba2bac5d335a4':
    raise ValueError('The existing backend differs from the reviewed version. It was preserved; share the error before continuing.')
if not 0 < WORKER_HOURS <= 3:
    raise ValueError('Use a worker time budget of at most three hours.')
print('Project:', ROOT)
print('Only targets: 1.5B / parent 17 and 123 / B training seed 2026 / mixed 1x.')
print('No new packages will be installed.')


## Cell 2 - prepare the targeted worker

Creates a separate helper; the original overnight file and training helpers are preserved.


In [ ]:
DRIVER_SOURCE = '"""Finish only mixed 1x, Qwen 1.5B, continuation seed 2026, parents 17 and 123."""\nimport argparse\nfrom collections import Counter\nfrom contextlib import ExitStack\nimport csv\nfrom datetime import datetime, timezone\nimport fcntl\nimport hashlib\nimport importlib.util\nimport json\nfrom pathlib import Path\nimport signal\nimport time\nimport traceback\n\nEXPECTED_BACKEND_SHA = \'d8101467d2037095bffdfb851959c2a2e2405daeb2de377b703ba2bac5d335a4\'\nSCOPE = \'Only mixed 1x / Qwen 1.5B / B training seed 2026 / parent seeds 17 and 123\'\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for chunk in iter(lambda: f.read(1024*1024), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef load_backend(root):\n    path = Path(root)/\'Method_B_Overnight.py\'\n    if sha(path) != EXPECTED_BACKEND_SHA:\n        raise ValueError(\'The existing Method_B_Overnight.py differs from the reviewed version; it was preserved.\')\n    spec = importlib.util.spec_from_file_location(\'finish_2026_backend\', path)\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    return module\n\n\ndef target_tasks(backend):\n    tasks = [t for t in backend.make_tasks() if t[\'setting\'] == \'mixed\' and t[\'budget\'] == 1\n             and (t[\'stage\'] == \'prepare\' or\n                  (t[\'model\'] == \'1.5B\' and t[\'training_seed\'] == 2026 and t[\'parent_seed\'] in (17, 123)))]\n    ids = {t[\'id\'] for t in tasks}\n    if len(tasks) != 7 or any(not set(t[\'dependencies\']) <= ids for t in tasks):\n        raise ValueError(\'Unexpected task graph; no experiment started.\')\n    if Counter(t[\'stage\'] for t in tasks) != {\'prepare\': 1, \'smoke\': 2, \'train\': 2, \'evaluate\': 2}:\n        raise ValueError(\'Only two target training/evaluation cases are permitted.\')\n    return tasks\n\n\ndef controller_class(n):\n    class FinishTwo(n.Controller):\n        def __init__(self, root, folder, hours):\n            super().__init__(root, folder, hours)\n            self.tasks = target_tasks(n)\n            self.area = root/\'outputs\'/n.STUDY/\'finish_1p5B_seed2026\'\n            self.meta.update(scope=SCOPE, expected_training_cases=2, expected_evaluation_cases=2,\n                external_requirements=[\'Shared evaluator/protection clearance\', \'Compatible A/C comparisons\',\n                    \'Transfer and other budgets are outside this two-run notebook\'])\n\n        def save(self, status=\'running\'):\n            self.meta[\'updated_utc\'] = datetime.now(timezone.utc).isoformat()\n            value = {\'status\': status, \'metadata\': self.meta, \'gpu\': self.gpu,\n                \'running_task\': self.running, \'tasks\': self.tasks,\n                \'capacity_checks\': list(self.capacities.values())}\n            n.atomic_json(self.folder/\'status.json\', value)\n            n.atomic_json(self.area/\'latest_status.json\', value)\n            self.report(status)\n\n        def report(self, status):\n            counts = Counter(t[\'status\'] for t in self.tasks)\n            train = sum(t[\'stage\'] == \'train\' and t[\'status\'] == \'completed\' for t in self.tasks)\n            evaluation = sum(t[\'stage\'] == \'evaluate\' and t[\'status\'] == \'completed\' for t in self.tasks)\n            lines = [\'METHOD B - TWO REMAINING MAIN RUNS\', SCOPE, \'Status: \' + status,\n                f\'Completed target training cases: {train}/2\',\n                f\'Completed target evaluation cases: {evaluation}/2\', \'Task counts: \' + json.dumps(counts),\n                \'Results remain preliminary development evidence.\', \'\']\n            for t in self.tasks:\n                lines.append(t[\'id\'] + \' | \' + t[\'status\'] + \' | \' + t.get(\'reason\', \'\'))\n            if self.meta.get(\'fatal_error\'):\n                lines += [\'\', \'ERROR: \' + self.meta[\'fatal_error\']]\n            lines += [\'\', \'Logs and target results: \' + str(self.folder),\n                \'All saved development results: \' + str(self.root/\'outputs\'/n.STUDY/\'development_results.csv\')]\n            n.atomic_text(self.folder/\'FINISH_REPORT.txt\', \'\\n\'.join(lines) + \'\\n\')\n            n.atomic_text(self.area/\'FINISH_REPORT.txt\', \'\\n\'.join(lines) + \'\\n\')\n\n        def preflight(self):\n            self.code = n.bootstrap(self.root)\n            reference = n.load_module(\'finish_reference\', self.code/\'run_method_b.py\')\n            reference.source(self.root)  # Existing pinned code and parent-reference checks.\n            work = self.root/\'outputs\'/n.STUDY/\'mixed/budget_1x\'\n            shared = self.root/\'outputs\'/n.STUDY/\'shared\'\n            bank_audit = n.read(shared/\'bank_audit.json\')\n            for name, key in [(\'mixed_candidate_bank.jsonl\', \'mixed_bank_sha256\'),\n                              (\'clean_development_reference.jsonl\', \'clean_development_sha256\'),\n                              (\'parent_length_bins.json\', \'length_bins_sha256\')]:\n                if n.sha(shared/name) != bank_audit[key]:\n                    raise ValueError(\'Existing primary-study input changed: \' + name)\n            cfg = n.read(self.root/\'b_workbench_500/config.json\')\n            if n.sha(self.root/\'data/train_n1000.jsonl\') != cfg[\'parent_manifest_sha256\']:\n                raise ValueError(\'Parent manifest changed\')\n            prepare = next(t for t in self.tasks if t[\'stage\'] == \'prepare\')\n            if not n.completed(self.root, prepare, {}):\n                raise ValueError(\'Existing 1x selection/label verification is missing or stale; no data were rebuilt.\')\n            registry = n.read(work/\'runtime/parent_registry.json\')\n            original_registry = n.read(self.root/\'b_workbench_500/parent_registry.json\')\n            if registry != original_registry:\n                raise ValueError(\'Runtime parent registry differs from the pinned original\')\n            for parent in (17, 123):\n                entry = registry[f\'1.5B_seed{parent}\']\n                if n.sha(self.root/entry[\'adapter_relative_path\']/\'adapter_model.safetensors\') != entry[\'expected_adapter_sha256\']:\n                    raise ValueError(\'Original parent adapter changed: \' + str(parent))\n            self.gpu = n.gpu_probe(self.root/\'venv_b_training_py312/bin/python\')\n            active = n.legacy_workers(self.root)\n            if active:\n                raise RuntimeError(\'Existing project training/evaluation workers are active: \' + json.dumps(active))\n            self.capacities = {(\'mixed\', 1): n.capacity(self.root, \'mixed\', 1)}\n            if self.capacities[(\'mixed\', 1)][\'infeasibility_proven\']:\n                raise ValueError(\'The existing 1x bank no longer satisfies necessary matching bounds\')\n            if self.work_deadline - time.time() < 180:\n                raise RuntimeError(\'Too little allocation time remains; use a fresh GPU session.\')\n            self.save(\'ready\')\n            print(\'STARTUP_PASSED. GPU:\', self.gpu[\'gpu\'], flush=True)\n            print(\'Only two main cases are scheduled: 1.5B / seed 2026 / parents 17 and 123.\', flush=True)\n            print(\'Existing verified smoke checks, training and predictions will be reused.\', flush=True)\n\n        def finish(self):\n            self.running = None\n            self.save(\'saving_results\')\n            if time.time() + 20 < self.hard_deadline:\n                result = n.supervise([self.root/\'venv_b_training_py312/bin/python\', \'-u\',\n                    self.code/\'run_method_b_eval_unattended.py\', \'summary\', \'--root\', self.root],\n                    self.folder/\'logs/final_summary.log\', self.hard_deadline - 5, stall_seconds=240)\n            else:\n                result = {\'return_code\': None, \'stop_reason\': \'Insufficient time for final summary\'}\n            self.meta[\'summary_result\'] = result\n            target_results = []\n            for task in self.tasks:\n                if task[\'stage\'] == \'evaluate\' and task[\'status\'] == \'completed\':\n                    if not n.completed(self.root, task, self.gpu):\n                        task.update(status=\'failed\', reason=\'Final canonical evidence recheck failed\')\n                        continue\n                    run = n.run_directory(self.root, task)\n                    saved = n.read(self.root/\'outputs\'/n.STUDY/\'mixed/budget_1x/evaluation\'/run.name/\'B_vs_parent.json\')\n                    if len(saved[\'comparisons\']) != 2:\n                        task.update(status=\'failed\', reason=\'Expected two benchmark comparisons\')\n                        continue\n                    target_results.extend(saved[\'comparisons\'])\n            n.atomic_json(self.folder/\'TARGET_RESULTS.json\', target_results)\n            if target_results:\n                with (self.folder/\'TARGET_RESULTS.csv\').open(\'w\', newline=\'\') as f:\n                    writer = csv.DictWriter(f, fieldnames=list(target_results[0]))\n                    writer.writeheader()\n                    writer.writerows(target_results)\n            passed = all(t[\'status\'] == \'completed\' for t in self.tasks) and len(target_results) == 4\n            summary_ok = result[\'return_code\'] == 0\n            receipt = {\'passed\': passed and summary_ok, \'target_training_and_evaluation_passed\': passed,\n                \'summary_updated\': summary_ok, \'scope\': SCOPE, \'target_comparisons\': len(target_results),\n                \'new_transfer_or_extra_budget_runs\': False, \'final_test_used\': False,\n                \'results_remain_preliminary\': True, \'session\': str(self.folder)}\n            n.atomic_json(self.folder/\'COMPLETION.json\', receipt)\n            self.save(\'completed\' if receipt[\'passed\'] else \'finished_with_remaining_work\')\n            print(\'TWO_RUN_FINISH_REPORT:\', json.dumps(receipt), flush=True)\n            print(\'TARGET_RESULTS:\', self.folder/\'TARGET_RESULTS.csv\', flush=True)\n            print(\'ALL_DEVELOPMENT_RESULTS:\', self.root/\'outputs\'/n.STUDY/\'development_results.csv\', flush=True)\n\n    return FinishTwo\n\n\ndef run(root, folder, hours):\n    root, folder = Path(root).resolve(), Path(folder).resolve()\n    folder.mkdir(parents=True, exist_ok=True)\n    n = load_backend(root)\n    n.STOP_REQUESTED = False\n    for sig in (signal.SIGINT, signal.SIGTERM):\n        signal.signal(sig, n.signal_stop)\n    controller = controller_class(n)(root, folder, hours)\n    try:\n        code = n.bootstrap(root)\n        recovery = n.load_module(\'finish_recovery\', code/\'evaluation_recovery.py\')\n        lock = root/\'outputs\'/n.STUDY/\'overnight/worker.lock\'\n        lock.parent.mkdir(parents=True, exist_ok=True)\n        with ExitStack() as stack:\n            handle = stack.enter_context(lock.open(\'a+\'))\n            try:\n                fcntl.flock(handle.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)\n            except BlockingIOError:\n                raise RuntimeError(\'Another overnight/finish worker is already running; no second worker started.\')\n            stack.enter_context(recovery.single_evaluation(root))\n            controller.preflight()\n            n.atomic_json(folder/\'startup.json\', {\'ready\': True, \'pid\': controller.meta[\'worker_pid\'],\n                \'gpu\': controller.gpu, \'scope\': SCOPE})\n            controller.queue()\n            controller.finish()\n        return 0 if n.read(folder/\'COMPLETION.json\')[\'passed\'] else 2\n    except BaseException as exc:\n        controller.meta.update(fatal_error=str(exc), traceback=traceback.format_exc())\n        controller.save(\'stopped_for_required_repair\')\n        n.atomic_json(folder/\'startup.json\', {\'ready\': False, \'error\': str(exc)})\n        traceback.print_exc()\n        return 1\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--root\', type=Path, required=True)\n    parser.add_argument(\'--run-folder\', type=Path, required=True)\n    parser.add_argument(\'--hours\', type=float, default=2.75)\n    args = parser.parse_args()\n    if not 0 < args.hours <= 3:\n        raise ValueError(\'This targeted runner accepts at most three worker hours.\')\n    raise SystemExit(run(args.root, args.run_folder, args.hours))\n'
DRIVER_SHA = hashlib.sha256(DRIVER_SOURCE.encode()).hexdigest()
DRIVER = ROOT / 'method_b_document_v3_code' / ('finish_1p5B_seed2026_' + DRIVER_SHA[:12] + '.py')
DRIVER.parent.mkdir(parents=True, exist_ok=True)
if DRIVER.exists() and DRIVER.read_text() != DRIVER_SOURCE:
    raise FileExistsError('A different helper already exists and was preserved: ' + str(DRIVER))
if not DRIVER.exists():
    DRIVER.write_text(DRIVER_SOURCE)
print('Targeted worker:', DRIVER)


## Cell 3 - start the two-run worker

Leave this cell running for up to one minute for startup checks. If startup takes
longer, Cell 4 shows progress. Do not launch the original full overnight queue.


In [ ]:
def start_two_runs():
    lock = ROOT / 'outputs/B_method_B_document_v3/overnight/worker.lock'
    lock.parent.mkdir(parents=True, exist_ok=True)
    with lock.open('a+') as handle:
        try:
            fcntl.flock(handle.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError:
            raise RuntimeError('Another project worker is running. No duplicate worker was started; wait for it to finish.')
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:6]
    folder = AREA / 'sessions' / stamp
    folder.mkdir(parents=True)
    log = folder / 'console.log'
    command = [str(TRAINING_PYTHON), '-u', str(DRIVER), '--root', str(ROOT),
               '--run-folder', str(folder), '--hours', str(WORKER_HOURS)]
    with log.open('w') as output:
        process = subprocess.Popen(command, stdout=output, stderr=subprocess.STDOUT,
            stdin=subprocess.DEVNULL, start_new_session=True, cwd=str(ROOT))
    (folder / 'launch.json').write_text(json.dumps({'pid': process.pid, 'driver_sha256': DRIVER_SHA,
        'run_folder': str(folder), 'command': command}, indent=2) + '\n')
    print('Worker PID:', process.pid)
    print('Console log:', log)
    print('Checking startup...')
    deadline = time.monotonic() + 60
    while time.monotonic() < deadline:
        startup = folder / 'startup.json'
        if startup.is_file():
            info = json.loads(startup.read_text())
            if not info.get('ready'):
                print(log.read_text(errors='replace')[-5000:])
                raise RuntimeError('Startup stopped. See the saved log above; no full queue was launched.')
            print('STARTUP PASSED:', info.get('gpu'))
            break
        if process.poll() is not None:
            print(log.read_text(errors='replace')[-5000:])
            raise RuntimeError('Worker exited during startup. Share the saved console log.')
        time.sleep(1)
    else:
        print('Startup is still in progress. Continue to the monitoring cell; do not start another worker.')
    return process, folder

if 'PROCESS' in globals() and PROCESS.poll() is None:
    print('This notebook already has a live worker. Continue monitoring:', RUN_FOLDER)
else:
    PROCESS, RUN_FOLDER = start_two_runs()


## Cell 4 - watch progress

This cell can show `[*]` for about 1-1.5 hours while the two cases run. Output is
refreshed every 15 seconds. The worker saves a heartbeat approximately every 30 seconds.
If you interrupt this monitoring cell, rerun **this cell** to resume watching.
The worker continues inside its active allocation.


In [ ]:
from IPython.display import clear_output

while True:
    clear_output(wait=True)
    status_path = RUN_FOLDER / 'status.json'
    if status_path.is_file():
        status = json.loads(status_path.read_text())
        print('Status:', status['status'])
        print('Updated:', status['metadata'].get('updated_utc'))
        active = status.get('running_task')
        if active:
            print('Current task:', active['id'])
            print('Task elapsed: %.1f minutes' % (active.get('elapsed_seconds', 0) / 60))
        for stage in ('train', 'evaluate'):
            done = sum(t['stage'] == stage and t['status'] == 'completed' for t in status['tasks'])
            print(stage + ':', str(done) + '/2 completed')
    log = RUN_FOLDER / 'console.log'
    if log.is_file():
        print('\nRecent console output:')
        print(log.read_text(errors='replace')[-2400:])
    print('\nSession folder:', RUN_FOLDER)
    if PROCESS.poll() is not None:
        print('Worker exit code:', PROCESS.returncode)
        break
    time.sleep(15)


## Cell 5 - verify completion and view the new results

Success requires `passed: true` in the completion receipt, **2/2 training and 2/2
evaluation cases**, and an updated shared summary. This cell does not start work.
If there is an error or deferred task, upload `FINISH_REPORT.txt` and `console.log`
from the printed session folder. Rerunning Cells 1-5 in a fresh allocation reuses
verified completions and the existing checkpoint/resume behavior.


In [ ]:
from IPython.display import Markdown, display

receipt_path = RUN_FOLDER / 'COMPLETION.json'
if not receipt_path.is_file():
    status_path = RUN_FOLDER / 'status.json'
    if status_path.is_file():
        print(json.dumps(json.loads(status_path.read_text()), indent=2)[-6000:])
    raise RuntimeError('No completion receipt yet. Check the monitoring cell and saved console.log.')
receipt = json.loads(receipt_path.read_text())
print(json.dumps(receipt, indent=2))
if receipt.get('passed') is not True:
    raise RuntimeError('Some work or summary generation remains. Upload FINISH_REPORT.txt and console.log; existing results are preserved.')
print('\nSUCCESS: both remaining 1.5B / seed-2026 cases are trained and evaluated.')
new_rows = json.loads((RUN_FOLDER / 'TARGET_RESULTS.json').read_text())
table = ['| Parent profile | Dataset | B seed | Parent accuracy | B accuracy | Change (pp) |',
         '|---|---|---:|---:|---:|---:|']
for r in new_rows:
    table.append(f"| {r['parent_profile']} | {r['dataset']} | {r['training_seed']} | "
                 f"{100*r['parent_accuracy']:.2f}% | {100*r['B_accuracy']:.2f}% | "
                 f"{r['change_percentage_points']:+.2f} |")
display(Markdown('\n'.join(table)))
print('New results:', RUN_FOLDER / 'TARGET_RESULTS.csv')
print('New completion receipt:', receipt_path)
print('All saved development comparisons:', ROOT / 'outputs/B_method_B_document_v3/development_results.csv')
print('Results remain preliminary; shared A/C comparisons, transfer and final evaluation are separate.')


## What to upload afterward

Save the notebook with Ctrl/Cmd+S. In Sol portal Files, open the session folder
printed above and download **COMPLETION.json** and **TARGET_RESULTS.json**.
Upload them here so the new results can be checked and added to the graphs.
Your canonical `development_results.csv` now also includes the newly completed cases.

This notebook updates result summaries; it does not create a new full model archive.
The new final adapters remain in the usual `mixed/budget_1x/runs/` directories.
